# predict.ipynb

**Langkah 5: inferensi pada gambar baru.**

Bergantung pada `data.ipynb` (dimuat dengan `%run`). Taruh gambar RGB ke `test_images/image/`; setiap gambar diubah ukurannya ke 512x512 dengan preprocessing yang sama seperti pelatihan, disegmentasi, dan mask prediksi diubah ukurannya kembali ke resolusi asli. Strip (Original | Masked) ditulis ke `test_images/mask/`.

In [1]:
# Muat modul yang dibutuhkan notebook ini.
# _DEMO=False menekan sel demo/self-check mereka selama %run.
_DEMO = False
%run data.ipynb
_DEMO = True

In [2]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import cv2
from glob import glob
from tqdm import tqdm
import tensorflow as tf

### Helper inferensi

In [3]:
def segment_image(model, image_bgr):
    """
    Jalankan model pada satu gambar BGR dengan ukuran apa pun. Mengembalikan
    mask biner (0/1) pada resolusi asli.
    """
    h, w = image_bgr.shape[:2]
    img = standardize_image(image_bgr)                  # -> 512x512, INTER_AREA
    x = np.expand_dims(img.astype(np.float32) / 255.0, axis=0)
    y = model.predict(x, verbose=0)[0]
    y = cv2.resize(y, (w, h))                            # kembali ke ukuran asli
    return (y > 0.5).astype(np.uint8)

In [4]:
def visualize_inference_results(results, save_dir="plots"):
    """
    Grid pasangan Original | Masked (dua pasangan per baris). Disimpan ke
    plots/09_inference_results.png. `results` adalah list berisi (orig_bgr, masked_bgr, nama).
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    if not results:
        return

    create_dir(save_dir)
    n = min(len(results), 8)
    pairs_per_row = 2
    cols = pairs_per_row * 2
    rows = int(np.ceil(n / pairs_per_row))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.4, rows * 3.4))
    axes = np.atleast_2d(axes)
    if axes.shape[0] == 1 and rows == 1:
        axes = axes.reshape(1, cols)

    for c, title in enumerate(["Original", "Masked"] * pairs_per_row):
        if c < axes.shape[1]:
            axes[0, c].set_title(title, fontsize=11, fontweight="bold", pad=6)

    for idx in range(rows * pairs_per_row):
        row = idx // pairs_per_row
        col_base = (idx % pairs_per_row) * 2
        for c in (col_base, col_base + 1):
            if c < axes.shape[1]:
                axes[row, c].axis("off")
        if idx < n:
            orig, masked, name = results[idx]
            axes[row, col_base].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
            axes[row, col_base].set_xlabel(name, fontsize=7)
            axes[row, col_base + 1].imshow(cv2.cvtColor(masked, cv2.COLOR_BGR2RGB))

    fig.suptitle("Hasil Inferensi - DeepLabV3+", fontsize=14, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(save_dir, "09_inference_results.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Hasil inferensi disimpan: {out}")

### Jalankan inferensi

Muat checkpoint dan segmentasikan setiap gambar di `test_images/image/`.

In [5]:
from IPython.display import Image, display

os.makedirs("test_images/image", exist_ok=True)
os.makedirs("test_images/mask", exist_ok=True)
create_dir("plots")

MODEL_PATH = os.path.join("files", "model.h5")
if not os.path.exists(MODEL_PATH):
    raise SystemExit("Tidak ada checkpoint. Jalankan train.ipynb terlebih dahulu.")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

paths = sorted(glob("test_images/image/*"))
print(f"Ditemukan {len(paths)} gambar.")

viz = []
for path in paths:
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        continue
    mask = segment_image(model, img)
    masked = (img * np.stack([mask] * 3, axis=-1)).astype(np.uint8)
    h = img.shape[0]
    divider = np.ones((h, 10, 3), np.uint8) * 128
    strip = np.concatenate([img, divider, masked], axis=1)
    name = os.path.splitext(os.path.basename(path))[0]
    cv2.imwrite(os.path.join("test_images", "mask", f"{name}.png"), strip)
    viz.append((img, masked, name))

if viz:
    visualize_inference_results(viz, save_dir="plots")
    display(Image("plots/09_inference_results.png"))
else:
    print("Tambahkan gambar ke test_images/image/ dan jalankan ulang.")

SystemExit: Tidak ada checkpoint. Jalankan train.ipynb terlebih dahulu.

c:\Users\manii\.conda\envs\tf\lib\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
